In [1]:
import os
import sys
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType
)

spark = SparkSession.builder \
  .appName("CDC-Bronze") \
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.lakehouse.type", "rest") \
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181") \
  .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
  .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000") \
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
  .config("spark.sql.defaultCatalog", "lakehouse") \
  .getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.1.0


In [4]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")
raw = spark.read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka:9092") \
  .option("subscribe", "dbserver1.public.customers") \
  .option("startingOffsets", "earliest") \
  .load()
from pyspark.sql import functions as F

In [9]:
raw_filtered = raw.filter(F.col("value").isNotNull())
bronze_df = raw_filtered.select(
  F.col("topic"),
  F.col("partition").alias("kafka_partition"),
  F.col("offset").alias("kafka_offset"),
  F.col("timestamp").alias("kafka_timestamp"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)
bronze_df.writeTo("lakehouse.cdc.bronze_customers").createOrReplace()
spark.sql("SELECT count(*) FROM lakehouse.cdc.bronze_customers").show()
spark.sql("""
  SELECT op, after_id, after_name, after_email, ts_ms
  FROM lakehouse.cdc.bronze_customers LIMIT 5
""").show(truncate=False)

+--------+
|count(1)|
+--------+
|     334|
+--------+

+---+--------+------------+-----------------------+-------------+
|op |after_id|after_name  |after_email            |ts_ms        |
+---+--------+------------+-----------------------+-------------+
|r  |7       |Grace Kim   |grace@example.com      |1777394349781|
|r  |92      |Oscar Garcia|oscar.garcia@inbox.org |1777394349793|
|r  |53      |Anna Muller |anna.muller@mail.com   |1777394349794|
|r  |68      |Noah Larsen |updated_68_274@mail.com|1777394349795|
|r  |21      |Chen Silva  |chen.silva@mail.com    |1777394349795|
+---+--------+------------+-----------------------+-------------+



In [10]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (
    id INT, name STRING, email STRING, country STRING, last_updated_ms BIGINT
  ) USING iceberg
""")

DataFrame[]

In [11]:
from pyspark.sql.window import Window
# Use COALESCE because for deletes, after_id is null but before_id has the key
bronze_with_key = bronze_df.withColumn(
  "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))
)

w = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").desc())
deduped = bronze_with_key \
  .filter(F.col("op").isNotNull()) \
  .withColumn("rn", F.row_number().over(w)) \
  .filter("rn = 1").drop("rn")

deduped.createOrReplaceTempView("cdc_batch")
spark.sql("""
  MERGE INTO lakehouse.cdc.silver_customers AS t
  USING cdc_batch AS s
  ON t.id = s.entity_id

  WHEN MATCHED AND s.op = 'd' THEN DELETE
  WHEN MATCHED AND s.op IN ('c','u','r') THEN UPDATE SET
    t.name = s.after_name, t.email = s.after_email,
    t.country = s.after_country, t.last_updated_ms = s.ts_ms

  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
    (id, name, email, country, last_updated_ms)
    VALUES (s.after_id, s.after_name, s.after_email, s.after_country, s.ts_ms)
""")

DataFrame[]

In [12]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT * FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)

+--------+
|count(1)|
+--------+
|     126|
+--------+

+---+--------------+------------------------+---------+---------------+
|id |name          |email                   |country  |last_updated_ms|
+---+--------------+------------------------+---------+---------------+
|7  |Grace Kim     |grace@example.com       |France   |1777394548253  |
|28 |Lena Novak    |updated_28_172@mail.com |France   |1777394349801  |
|29 |Yuki Tanaka   |updated_29_403@test.net |Lithuania|1777394349806  |
|30 |Mateo Andersen|updated_30_513@inbox.org|Brazil   |1777394349817  |
|31 |Alex Tanaka   |updated_31_686@test.net |Brazil   |1777394951618  |
+---+--------------+------------------------+---------+---------------+

